# Analyse exploratoire et nettoyage des données Valmont

Objectif : vérifier la qualité des deux fichiers fournis, documenter chaque décision de nettoyage et produire le fichier propre commun `data/clean/valmont_clean.csv`, point de départ de toutes les questions.

Responsable : Kadidiatou Ibrahima Bagayoko

In [1]:
import pandas as pd
import numpy as np

# Chemins relatifs au dossier notebooks/
CHEMIN_RAW = "../data/raw/"
CHEMIN_CLEAN = "../data/clean/"

reservations = pd.read_csv(CHEMIN_RAW + "Valmont_reservations.csv")
hotels = pd.read_csv(CHEMIN_RAW + "Valmont_hotels.csv")

print("Réservations :", reservations.shape)
print("Hôtels :", hotels.shape)
reservations.head()


Réservations : (940, 13)
Hôtels : (32, 6)


,reservation_id,hotel_id,segment_declare,nb_nuits,nb_personnes,delai_reservation_j,mois_sejour,weekend,canal,prix_nuit_eur,petit_dejeuner,nuitees_cumulees_client,a_annule
0,R700357,H45,groupe,1,11.0,15.0,9,non,OTA,80.33,non,9.0,non
1,R700658,H51,loisir,2,4.0,9.0,8,oui,direct,157.02,oui,4.0,non
2,R700649,H41,affaires,5,1.0,7.0,12,oui,direct,434.92,oui,6.0,non
3,R700732,H69,loisir,4,1.0,83.0,2,non,OTA,92.70,non,1.0,oui
4,R700098,H52,loisir,3,6.0,213.0,3,oui,OTA,85.74,oui,8.0,non


## 1. Structure des fichiers et cohérence de la jointure

In [2]:
# Types de chaque colonne et nombre de valeurs renseignées
reservations.info()

# Valeurs manquantes par colonne (seules les colonnes concernées sont affichées)
manquants = reservations.isna().sum()
print("\nValeurs manquantes :\n", manquants[manquants > 0])

# Jointure : chaque hôtel des réservations doit exister dans le référentiel
hotels_inconnus = set(reservations["hotel_id"]) - set(hotels["hotel_id"])
print("\nHôtels absents du référentiel :", hotels_inconnus)
print("Identifiants d'hôtel en double :", hotels["hotel_id"].duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 940 entries, 0 to 939
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   reservation_id           940 non-null    str    
 1   hotel_id                 940 non-null    str    
 2   segment_declare          940 non-null    str    
 3   nb_nuits                 940 non-null    int64  
 4   nb_personnes             916 non-null    float64
 5   delai_reservation_j      940 non-null    float64
 6   mois_sejour              940 non-null    int64  
 7   weekend                  940 non-null    str    
 8   canal                    940 non-null    str    
 9   prix_nuit_eur            940 non-null    float64
 10  petit_dejeuner           916 non-null    str    
 11  nuitees_cumulees_client  940 non-null    float64
 12  a_annule                 940 non-null    str    
dtypes: float64(4), int64(2), str(7)
memory usage: 95.6 KB

Valeurs manquantes :
 nb_personnes  

## 2. Doublons

In [3]:
nb_doublons_exacts = reservations.duplicated().sum()
nb_ids_en_double = reservations["reservation_id"].duplicated().sum()
print("Lignes strictement identiques :", nb_doublons_exacts)
print("Identifiants de réservation répétés :", nb_ids_en_double)

# Si les deux chiffres sont égaux, chaque identifiant répété correspond
# à une copie exacte : aucun conflit d'information entre deux versions
reservations = reservations.drop_duplicates().reset_index(drop=True)

print("Lignes après dédoublonnage :", len(reservations))
print("Identifiants uniques :", reservations["reservation_id"].is_unique)
print("Valeurs manquantes restantes :\n", reservations.isna().sum()[lambda s: s > 0])

Lignes strictement identiques : 140
Identifiants de réservation répétés : 140
Lignes après dédoublonnage : 800
Identifiants uniques : True
Valeurs manquantes restantes :
 nb_personnes      20
petit_dejeuner    20
dtype: int64


**Décision** : 140 lignes sont des copies exactes d'autres réservations (même identifiant, mêmes valeurs dans toutes les colonnes). Elles sont supprimées, car les garder compterait deux fois les mêmes clients. Le fichier passe de 940 à 800 réservations, alors que la fiche du sujet en annonçait 940.

## 3. Valeurs manquantes

In [ ]:
manque_pers = reservations["nb_personnes"].isna()
manque_pdj = reservations["petit_dejeuner"].isna()

print("Lignes où les deux manquent :", (manque_pers & manque_pdj).sum())
print("\nSegments des nb_personnes manquants :\n", reservations.loc[manque_pers, "segment_declare"].value_counts())

# Le manque est-il lié à l'annulation ? Si oui, il porte de l'information
taux = lambda s: (s == "oui").mean().round(3)
print("\nTaux d'annulation selon que nb_personnes manque :\n", reservations.groupby(manque_pers)["a_annule"].apply(taux))
print("\nTaux d'annulation selon que petit_dejeuner manque :\n", reservations.groupby(manque_pdj)["a_annule"].apply(taux))
print("\nMédiane de nb_personnes par segment :\n", reservations.groupby("segment_declare")["nb_personnes"].median())

In [ ]:
# On garde la trace du manque : sur 20 lignes l'écart d'annulation n'est pas
# significatif, mais il reste une information que les modèles pourront tester
reservations["nb_personnes_manquant"] = manque_pers.astype(int)
reservations["petit_dejeuner_manquant"] = manque_pdj.astype(int)

# Imputation par la médiane du segment : la taille d'un groupe (11) n'a rien
# à voir avec celle d'un voyage d'affaires (1), une médiane globale fausserait les deux
reservations["nb_personnes"] = reservations["nb_personnes"].fillna(
    reservations.groupby("segment_declare")["nb_personnes"].transform("median")
)
reservations["petit_dejeuner"] = reservations["petit_dejeuner"].fillna("inconnu")

print(reservations.isna().sum().sum(), "valeur manquante restante")

**Décision** : 20 valeurs de `nb_personnes` et 20 de `petit_dejeuner` manquent, dont une seule ligne commune. Le nombre de personnes est imputé par la médiane de son segment déclaré, car les tailles diffèrent fortement entre segments (1 en affaires, 11 en groupe). Le petit-déjeuner manquant devient une modalité « inconnu » plutôt qu'une supposition. Deux colonnes indicatrices conservent la trace du manque : les réservations incomplètes s'annulent plus souvent (35 à 40 % contre 26 %), sans que l'écart soit significatif sur 20 lignes.